# 02.5_build_dataset

This notebook builds a subject-level sleep feature dataset from multiple hypnogram files.

Pipeline:
- load many hypnogram CSV files
- extract subject-level sleep features
- combine all rows into one dataset
- save `sleep_features_all_subjects.csv`

Output:
- participant-level feature table ready for downstream MESA merge

In [4]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    raise FileNotFoundError('Project root not found. Make sure README.md and data/ exist.')

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR =', RAW_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

csv_files = [RAW_DIR / 'sleep_edf' / 'hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

PROJECT_ROOT = C:\Users\vi\sleep-neuroscience-project
RAW_DIR = C:\Users\vi\sleep-neuroscience-project\data\raw
PROCESSED_DIR = C:\Users\vi\sleep-neuroscience-project\data\processed
Found 1 CSV files
hypno_df.csv


In [2]:
INPUT_DIR = Path('hypnograms')
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [3]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:10]:
    print(f.name)

Found 0 CSV files


In [4]:
from pathlib import Path

INPUT_DIR = Path('hypnograms')
INPUT_DIR.mkdir(exist_ok=True)

print("Created or already exists:", INPUT_DIR.resolve())

Created or already exists: C:\Users\vi\hypnograms


In [5]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

Found 0 CSV files


In [6]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
for p in Path('.').iterdir():
    print(p)

Current working directory:
C:\Users\vi

Files and folders here:
.astropy
.config
.ipynb_checkpoints
.ipython
.jupyter
.matplotlib
.ms-ad
.python_history
01_data_exploration.ipynb
02.5_build_dataset.ipynb
02_feature_engineering.ipynb
03_ml_pipeline.ipynb
AppData
Apple
Application Data
Contacts
Cookies
Desktop
Documents
Downloads
Favorites
hypnograms
hypno_df.csv
Links
Local Settings
mne_data
Music
NetHood
NTUSER.DAT
ntuser.dat.LOG1
ntuser.dat.LOG2
NTUSER.DAT{34c8ebeb-b9ef-11ef-b986-98bd80157f92}.TM.blf
NTUSER.DAT{34c8ebeb-b9ef-11ef-b986-98bd80157f92}.TMContainer00000000000000000001.regtrans-ms
NTUSER.DAT{34c8ebeb-b9ef-11ef-b986-98bd80157f92}.TMContainer00000000000000000002.regtrans-ms
ntuser.ini
OneDrive
PCManger
Pictures
PrintHood
pulsar-navigation-project
pulsar-navigation-project.zip
Recent
Saved Games
Searches
SendTo
sleep_features_subject_001.csv
Videos
WPS Cloud Files
главное меню
Мои документы
Шаблоны


In [7]:
INPUT_DIR = RAW_DIR / 'sleep_edf'
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [8]:
csv_files = [RAW_DIR / 'sleep_edf' / 'hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

Found 1 CSV files
hypno_df.csv


In [9]:
test_file = csv_files[0]
print('Testing file:', test_file.name)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
test_features

Testing file: hypno_df.csv


NameError: name 'load_hypnogram_csv' is not defined

In [10]:
INPUT_DIR = Path('hypnograms')
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [11]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))

print(f'Found {len(csv_files)} CSV files')
for f in csv_files[:10]:
    print(f.name)

Found 0 CSV files


In [12]:
def clean_hypnogram(df):
    df = df.copy()
    df = df[~df['description'].str.contains('?', regex=False, na=False)].copy()
    df = df[df['duration'] > 0].copy()
    df = df.reset_index(drop=True)
    return df

In [13]:
def find_sleep_boundaries(df):
    sleep_mask = df['description'] != 'Sleep stage W'

    if sleep_mask.sum() == 0:
        return None, None

    first_sleep_idx = df[sleep_mask].index[0]
    last_sleep_idx = df[sleep_mask].index[-1]

    return first_sleep_idx, last_sleep_idx

In [14]:
def extract_sleep_features(hypno_df, subject_id=None):
    df = clean_hypnogram(hypno_df)

    total_recording_time_sec = df['duration'].sum()

    first_sleep_idx, last_sleep_idx = find_sleep_boundaries(df)

    if first_sleep_idx is None:
        return pd.DataFrame([{
            'subject_id': subject_id,
            'recording_hours': total_recording_time_sec / 3600,
            'sleep_period_hours': np.nan,
            'total_sleep_hours': 0,
            'sleep_latency_min': np.nan,
            'rem_latency_min': np.nan,
            'fragmentation': np.nan,
            'wake_pct_in_sleep_period': np.nan,
            'rem_pct_of_tst': np.nan,
            'n1_pct_of_tst': np.nan,
            'n2_pct_of_tst': np.nan,
            'deep_sleep_pct_of_tst': np.nan,
            'sleep_efficiency': np.nan
        }])

    sleep_latency_sec = df.loc[:first_sleep_idx - 1, 'duration'].sum() if first_sleep_idx > 0 else 0

    rem_indices = df[df['description'] == 'Sleep stage R'].index
    rem_after_sleep = rem_indices[rem_indices >= first_sleep_idx]

    if len(rem_after_sleep) > 0:
        first_rem_idx = rem_after_sleep[0]
        rem_latency_sec = df.loc[first_sleep_idx:first_rem_idx - 1, 'duration'].sum() if first_rem_idx > first_sleep_idx else 0
    else:
        rem_latency_sec = np.nan

    sleep_period_df = df.loc[first_sleep_idx:last_sleep_idx].copy().reset_index(drop=True)
    sleep_period_time_sec = sleep_period_df['duration'].sum()

    wake_in_sleep_period_sec = sleep_period_df.loc[
        sleep_period_df['description'] == 'Sleep stage W', 'duration'
    ].sum()

    total_sleep_time_sec = sleep_period_time_sec - wake_in_sleep_period_sec

    sleep_only_df = sleep_period_df[sleep_period_df['description'] != 'Sleep stage W'].copy()
    sleep_stage_time = sleep_only_df.groupby('description')['duration'].sum()

    if total_sleep_time_sec > 0:
        sleep_stage_pct = sleep_stage_time / total_sleep_time_sec * 100
    else:
        sleep_stage_pct = pd.Series(dtype=float)

    fragmentation = (sleep_period_df['description'] != sleep_period_df['description'].shift()).sum()

    features = {
        'subject_id': subject_id,
        'recording_hours': total_recording_time_sec / 3600,
        'sleep_period_hours': sleep_period_time_sec / 3600,
        'total_sleep_hours': total_sleep_time_sec / 3600,
        'sleep_latency_min': sleep_latency_sec / 60,
        'rem_latency_min': rem_latency_sec / 60 if pd.notna(rem_latency_sec) else np.nan,
        'fragmentation': fragmentation,
        'wake_pct_in_sleep_period': (
            wake_in_sleep_period_sec / sleep_period_time_sec * 100
            if sleep_period_time_sec > 0 else np.nan
        ),
        'rem_pct_of_tst': sleep_stage_pct.get('Sleep stage R', 0),
        'n1_pct_of_tst': sleep_stage_pct.get('Sleep stage 1', 0),
        'n2_pct_of_tst': sleep_stage_pct.get('Sleep stage 2', 0),
        'deep_sleep_pct_of_tst': (
            sleep_stage_pct.get('Sleep stage 3', 0) +
            sleep_stage_pct.get('Sleep stage 4', 0)
        ),
        'sleep_efficiency': (
            total_sleep_time_sec / sleep_period_time_sec
            if sleep_period_time_sec > 0 else np.nan
        )
    }

    return pd.DataFrame([features])

In [15]:
def load_hypnogram_csv(file_path):
    df = pd.read_csv(file_path)

    if 'duration' not in df.columns and 'duration_sec' in df.columns:
        df = df.rename(columns={'duration_sec': 'duration'})

    required_cols = {'description', 'duration'}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    df = df[['description', 'duration']].copy()
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')

    return df

In [16]:
def parse_subject_id(file_path):
    return file_path.stem

In [17]:
test_file = csv_files[0]
print('Testing file:', test_file.name)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
test_features

IndexError: list index out of range

In [18]:
from pathlib import Path

INPUT_DIR = RAW_DIR / 'sleep_edf'
OUTPUT_FEATURES_PATH = PROCESSED_DIR / 'sleep_features_all_subjects.csv'
OUTPUT_LOG_PATH = PROCESSED_DIR / 'processing_log.csv'

In [19]:
csv_files = [RAW_DIR / 'sleep_edf' / 'hypno_df.csv']

print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f.name)

Found 1 CSV files
hypno_df.csv


In [20]:
if len(csv_files) == 0:
    print("No CSV files found. Re-run the path/file discovery cells.")
else:
    test_file = csv_files[0]
    print('Testing file:', test_file.name)

    test_hypno_df = load_hypnogram_csv(test_file)
    display(test_hypno_df.head())

    test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
    display(test_features)

Testing file: hypno_df.csv


,description,duration
0,Sleep stage W,30630.0
1,Sleep stage 1,120.0
2,Sleep stage 2,390.0
3,Sleep stage 3,30.0
4,Sleep stage 2,30.0


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


In [21]:
all_features = []
processing_log = []

for file_path in csv_files:
    subject_id = parse_subject_id(file_path)

    try:
        hypno_df = load_hypnogram_csv(file_path)
        features_df = extract_sleep_features(hypno_df, subject_id=subject_id)

        all_features.append(features_df)

        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'success',
            'n_rows': len(hypno_df),
            'message': ''
        })

    except Exception as e:
        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'failed',
            'n_rows': np.nan,
            'message': str(e)
        })

sleep_features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
processing_log_df = pd.DataFrame(processing_log)

In [22]:
print('sleep_features_df shape:', sleep_features_df.shape)
display(sleep_features_df.head())

print('processing_log_df shape:', processing_log_df.shape)
display(processing_log_df.head(20))

sleep_features_df shape: (1, 13)


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


processing_log_df shape: (1, 5)


,subject_id,file_name,status,n_rows,message
0,hypno_df,hypno_df.csv,success,153,


In [23]:
processing_log_df['status'].value_counts(dropna=False)

status
success    1
Name: count, dtype: int64

In [24]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

,subject_id,file_name,status,n_rows,message


In [25]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

,subject_id,file_name,status,n_rows,message


In [26]:
from pathlib import Path

INPUT_DIR = RAW_DIR / 'sleep_edf'
csv_files = [RAW_DIR / 'sleep_edf' / 'hypno_df.csv']

print("csv_files:", csv_files)
print("len(csv_files):", len(csv_files))

csv_files: [WindowsPath('hypno_df.csv')]
len(csv_files): 1


In [27]:
test_file = csv_files[0]
print("Testing file:", test_file)

test_hypno_df = load_hypnogram_csv(test_file)
display(test_hypno_df.head())

test_features = extract_sleep_features(test_hypno_df, subject_id=parse_subject_id(test_file))
display(test_features)

Testing file: hypno_df.csv


,description,duration
0,Sleep stage W,30630.0
1,Sleep stage 1,120.0
2,Sleep stage 2,390.0
3,Sleep stage 3,30.0
4,Sleep stage 2,30.0


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


In [28]:
all_features = []
processing_log = []

for file_path in csv_files:
    subject_id = parse_subject_id(file_path)

    try:
        hypno_df = load_hypnogram_csv(file_path)
        features_df = extract_sleep_features(hypno_df, subject_id=subject_id)

        all_features.append(features_df)

        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'success',
            'n_rows': len(hypno_df),
            'message': ''
        })

    except Exception as e:
        processing_log.append({
            'subject_id': subject_id,
            'file_name': file_path.name,
            'status': 'failed',
            'n_rows': np.nan,
            'message': str(e)
        })

sleep_features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
processing_log_df = pd.DataFrame(processing_log)

print("sleep_features_df shape:", sleep_features_df.shape)
print("processing_log_df shape:", processing_log_df.shape)

display(sleep_features_df)
display(processing_log_df)

sleep_features_df shape: (1, 13)
processing_log_df shape: (1, 5)


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


,subject_id,file_name,status,n_rows,message
0,hypno_df,hypno_df.csv,success,153,


In [29]:
print('sleep_features_df shape:', sleep_features_df.shape)
display(sleep_features_df.head())

print('processing_log_df shape:', processing_log_df.shape)
display(processing_log_df.head(20))

sleep_features_df shape: (1, 13)


,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


processing_log_df shape: (1, 5)


,subject_id,file_name,status,n_rows,message
0,hypno_df,hypno_df.csv,success,153,


In [30]:
processing_log_df['status'].value_counts(dropna=False)

status
success    1
Name: count, dtype: int64

In [31]:
failed_df = processing_log_df[processing_log_df['status'] == 'failed'].copy()
failed_df

,subject_id,file_name,status,n_rows,message


In [32]:
sleep_features_df.to_csv(PROCESSED_DIR / 'sleep_features_all_subjects.csv', index=False)
processing_log_df.to_csv(PROCESSED_DIR / 'processing_log.csv', index=False)

In [33]:
check_df = pd.read_csv('sleep_features_all_subjects.csv')
check_df

,subject_id,recording_hours,sleep_period_hours,total_sleep_hours,sleep_latency_min,rem_latency_min,fragmentation,wake_pct_in_sleep_period,rem_pct_of_tst,n1_pct_of_tst,n2_pct_of_tst,deep_sleep_pct_of_tst,sleep_efficiency
0,hypno_df,22.083333,6.008333,5.441667,510.5,89.0,151,9.431345,19.14242,8.882083,38.284839,33.690658,0.905687


NameError: name 'features_df' is not defined